<a href="https://colab.research.google.com/github/NouraAbuthnain/slm-math-reasoning-agent/blob/main/slm_code_reasoning_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Install Dependencies

In [ ]:
!pip install -q "unsloth[colab-new]" xformers transformers trl datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

2. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only
import os
import json

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


3. Load Base Model

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


4. Baseline Inference (Before Training)

In [ ]:
print("===== BEFORE TRAINING =====")
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": "A deep-sea monster rises from the waters once every hundred years to feast on a ship and sate its hunger. Over three hundred years, it has consumed 847 people. Ships have been built larger over time, so each new ship has twice as many people as the last ship. How many people were on the ship the monster ate in the first hundred years?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=300,
    use_cache=False, # Changed from True to False
    temperature=0.7,
    top_p=0.9,
)

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(response)

===== BEFORE TRAINING =====


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To solve this problem, we need to determine the number of people on the ship that the monster ate during the first hundred years.

Given:
- The monster eats ships that have increased in size.
- Each new ship has twice as many people as the previous one.
- The total number of people eaten over 300 years is 847.
- We need to find out how many people were on the ship the monster ate in the first hundred years.

Let's denote the number of people on the first ship (in the first hundred years) as \( x \). According to the problem, each subsequent ship will have twice the number of people as the previous one. Therefore, the second ship will have \( 2x \), the third ship will have \( 4x \), and so on.

The total number of people eaten over 300 years can be represented as an infinite geometric series with the first term being \( x \) and common ratio \( 2 \):

\[ x + 2x + 4x + 8x + ... = 847 \]

This series sums up to:

\[ x(1 + 2 + 4 + 8 + ...) = 847 \]

We know that the sum of an infinite geo

5. Load Dataset, Train / Validation Split

In [ ]:
raw_dataset = load_dataset("donghuna/generated_code-gsm8k-plan", split="train")
print(raw_dataset.column_names)
print(raw_dataset[0])

# Shuffle once for reproducibility
raw_dataset = raw_dataset.shuffle(seed=42)

# Optional: make train/validation split
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

README.md:   0%|          | 0.00/349 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/511k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

['question', 'answer', 'solution']
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'solution': '```python\n# Define variables for April and May sales\napril_sales = 48\nmay_sales = 0\n\n# Calculate May sales\nmay_sales = april_sales // 2\n\n# Calculate total sales\ntotal_sales = may_sales + april_sales\n\nprint(f"Total Sales: {total_sales}")\n```'}


6. Add LoRA Adapters (QLoRA)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    max_seq_length=max_seq_length,
)

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


7. Dataset Formatting

In [ ]:
def formatting_prompts_func(examples):
    texts = []

    questions = examples["question"]
    answers = examples["answer"]
    solutions = examples["solution"]

    for q, a, s in zip(questions, answers, solutions):
        # Recommended: teach reasoning + final answer
        assistant_text = f"""Plan and solution:
{s}

Final Answer:
{a}"""

        convo = [
            {"role": "user", "content": q},
            {"role": "assistant", "content": assistant_text},
        ]

        formatted_text = tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(formatted_text)

    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)

# Keep only text column
cols_to_remove_train = [c for c in train_dataset.column_names if c != "text"]
cols_to_remove_eval = [c for c in eval_dataset.column_names if c != "text"]

train_dataset = train_dataset.remove_columns(cols_to_remove_train)
eval_dataset = eval_dataset.remove_columns(cols_to_remove_eval)

print(train_dataset.column_names)
print(train_dataset[0]["text"])

Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

['text']
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Jeff committed to run for an hour a day during weekdays. On Thursday, he cut short his run by 20 minutes but was able to jog 10 minutes more on Friday. How many minutes was he able to run for that week?<|im_end|>
<|im_start|>assistant
Plan and solution:
```python
# Define variables
monday_to_wednesday_minutes = 60 * 3 # Total minutes from Monday to Wednesday
thursday_minutes = 60 - 20          # Minutes ran on Thursday
friday_minutes = 60 + 10            # Minutes ran on Friday
total_minutes = monday_to_wednesday_minutes + thursday_minutes + friday_minutes
print(total_minutes)                # Output: 980
```

Final Answer:
Jeff was able to run a total of 60 x 3 = <<60*3=180>>180 minutes from Monday to Wednesday.
On Thursday, Jeff ran for 60 - 20 = <<60-20=40>>40 minutes only.
On Friday, he ran for 60 + 10 = <<60+10=70>>70 minutes.
Therefore, Jeff ran a total of 1

8. Training Setup

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        output_dir="outputs",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)

# Train only on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Debug one sample
i = 5
print("FULL INPUT:")
print(tokenizer.decode(trainer.train_dataset[i]["input_ids"]))

print("\nMASKED LABELS VIEW:")
space = tokenizer(" ", add_special_tokens=False)["input_ids"][0]
labels_view = [space if x == -100 else x for x in trainer.train_dataset[i]["labels"]]
print(tokenizer.decode(labels_view))

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/950 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/950 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/950 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

FULL INPUT:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Lily has 5 lottery tickets to sell.  She sells the first ticket for $1.  She then sells each successive ticket for a dollar more than the previous ticket. She plans to keep a $4 profit and give the remaining money as the prize. How much money will the winner of the lottery receive?<|im_end|>
<|im_start|>assistant
Plan and solution:
```python
# Step 1: Define variables and initial values
n = 5 # Number of tickets sold
price = 1 # Initial price of the first ticket
profit = 4 # Profit kept by Lily

# Step 2: Calculate the total money collected
total_money = n * (price + profit)
print(f"Total Money Collected: {total_money}")

# Step 3: Calculate the total prize money
prize_money = total_money - profit
print(f"Prize Money: {prize_money}")

# Step 4: Output the total prize money
print("The winner of the lottery receives", prize_money)
```

Final Answer:
The second ti

9. Train the Model

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 950 | Num Epochs = 3 | Total steps = 357
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Epoch,Training Loss,Validation Loss
1,0.329969,0.363180
2,0.306637,0.358249
3,0.235300,0.377070


10. Inference After Training

In [ ]:
print("===== AFTER TRAINING =====")
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=300,
    use_cache=False,
    temperature=0.7,
    top_p=0.9,
)

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(response)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== AFTER TRAINING =====
Plan and solution:
```python
# Define variables for the problem

# Step 1: Initialize the number of clips sold in April and May
clips_sold_in_april = 48
clips_sold_in_may = None # unknown at this point

# Step 2: Calculate the number of clips sold in May
clips_sold_in_may = clips_sold_in_april / 2

# Step 3: Calculate the total number of clips sold in both months
total_clips_sold = clips_sold_in_april + clips_sold_in_may

print(f"Total number of clips sold in {total_clips_sold}")
```

Final Answer:
In May, Natalia sold 1/2 * 48 clips = <<1/2*48=24>>24 clips.
Altogether in April and May, Natalia sold 48 clips + 24 clips = <<48+24=72>>72 clips.
#### 72


11. Save Model

In [ ]:
model.save_pretrained("lora_plan_model")
tokenizer.save_pretrained("lora_plan_model")

('lora_plan_model/tokenizer_config.json',
 'lora_plan_model/chat_template.jinja',
 'lora_plan_model/tokenizer.json')

12. LLM-as-a-Judge Evaluation (DeepSeek)

In [ ]:
!pip install -q openai
from openai import OpenAI
from google.colab import userdata

os.environ["DEEPSEEK_API_KEY"] = userdata.get('api_key')

In [ ]:
client = OpenAI(api_key=os.environ.get('DEEPSEEK_API_KEY'), base_url="https://api.deepseek.com")

JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator for math solutions designed for high school students.

Evaluate the model answer based on:

1. correctness_score (1-5)
2. reasoning_score (1-5)
3. clarity_score (1-5)
4. student_friendliness_score (1-5)
5. final_verdict ("pass" or "fail")

Return ONLY JSON in this format:

{
  "correctness_score": 0,
  "reasoning_score": 0,
  "clarity_score": 0,
  "student_friendliness_score": 0,
  "final_verdict": "pass or fail",
  "short_reason": "brief explanation"
}
"""

def judge_with_deepseek(question, gold_answer, model_prediction):

    prompt = f"""
Evaluate the following math solution.

Question:
{question}

Reference Answer:
{gold_answer}

Model Answer:
{model_prediction}
"""

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        stream=False
    )

    content = response.choices[0].message.content

    # Strip markdown code block fences if present
    if content.startswith('```json') and content.endswith('```'):
        content = content[len('```json'):-len('```')].strip()

    try:
        return json.loads(content)
    except:
        return {"error": "Invalid JSON", "raw_output": content}

In [ ]:
question = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
gold_answer = "#### 72"
prediction = "She sold 48 + 24 = 72 clips."

result = judge_with_deepseek(question, gold_answer, prediction)

print(result)

{'correctness_score': 5, 'reasoning_score': 5, 'clarity_score': 5, 'student_friendliness_score': 5, 'final_verdict': 'pass', 'short_reason': 'The answer is correct, clearly shows the reasoning (half of 48 is 24, then sum), and is presented in a simple, direct way that a student can easily follow.'}


In [ ]:
def generate_response(model, tokenizer, question):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=300,
        use_cache=False,
        temperature=0.7,
        top_p=0.9,
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

prediction = generate_response(model, tokenizer, question)

judge_result = judge_with_deepseek(
    question,
    gold_answer,
    prediction
)

print("Prediction:")
print(prediction)

print("\nJudge result:")
print(judge_result)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prediction:
Plan and solution:
```python
# Define variables for the problem
clips_sold_april = 48 # Number of clips sold in April
clips_sold_may = None # Number of clips sold in May

# Calculate the number of clips sold in May
clips_sold_may = clips_sold_april / 2

# Calculate the total number of clips sold in both months
total_clips_sold = clips_sold_april + clips_sold_may

print(f"Total number of clips sold: {total_clips_sold}")
```

Final Answer:
In May, Natalia sold 1/2 * 48 = <<1/2*48=24>>24 clips.
Altogether, Natalia sold 48 + 24 = <<48+24=72>>72 clips in April and May.
#### 72

Judge result:
{'correctness_score': 5, 'reasoning_score': 5, 'clarity_score': 5, 'student_friendliness_score': 5, 'final_verdict': 'pass', 'short_reason': 'The solution correctly interprets the problem, uses clear steps, explains calculations in plain language, and arrives at the correct answer of 72.'}


In [ ]:
eval_subset = raw_dataset.select(range(5))

results = []

for sample in eval_subset:
    question = sample["question"]
    gold = sample["answer"]

    pred = generate_response(model, tokenizer, question)

    judge = judge_with_deepseek(question, gold, pred)

    results.append({
        "question": question,
        "prediction": pred,
        "judge": judge
    })

for i, r in enumerate(results):
    print(f"\n=== Sample {i+1} ===")
    print("Question:", r["question"])
    print("Prediction:", r["prediction"])
    print("Judge:", r["judge"])

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


=== Sample 1 ===
Question: Stephen has 110 ants in his ant farm.  Half of the ants are worker ants, 20 percent of the worker ants are male.  How many female worker ants are there?
Prediction: Plan and solution:
```python
# Define variables
total_ants = 110
worker_percentage = 0.5 # 50%
male_worker_percentage = 0.2 # 20%

# Calculate the number of worker ants
num_of_worker_ants = int(total_ants * worker_percentage)

# Calculate the number of male worker ants
num_of_male_worker_ants = int(num_of_worker_ants * male_worker_percentage)

# Calculate the number of female worker ants
num_of_female_worker_ants = num_of_worker_ants - num_of_male_worker_ants

print(f"There are {num_of_female_worker_ants} female worker ants.")
```

Final Answer:
Worker ants:110/2=<<110/2=55>>55
Male worker ants:55(.20)=<<55*.20=11>>11
Female worker ants:55-11=<<55-11=44>>44
#### 44
Judge: {'correctness_score': 5, 'reasoning_score': 5, 'clarity_score': 5, 'student_friendliness_score': 4, 'final_verdict': 'pass', '

In [ ]:
def average(scores):
    return sum(scores) / len(scores) if scores else 0

summary = {
    "num_samples": len(results),
    "avg_correctness_score": average([r["judge"]["correctness_score"] for r in results if "correctness_score" in r["judge"]]),
    "avg_reasoning_score": average([r["judge"]["reasoning_score"] for r in results if "reasoning_score" in r["judge"]]),
    "avg_clarity_score": average([r["judge"]["clarity_score"] for r in results if "clarity_score" in r["judge"]]),
    "avg_student_friendliness_score": average([r["judge"]["student_friendliness_score"] for r in results if "student_friendliness_score" in r["judge"]]),
    "pass_rate": sum(r["judge"].get("final_verdict") == "pass" for r in results) / len(results),
}

print(summary)

{'num_samples': 5, 'avg_correctness_score': 3.2, 'avg_reasoning_score': 3.2, 'avg_clarity_score': 3.6, 'avg_student_friendliness_score': 3.2, 'pass_rate': 0.6}


In [ ]:
with open("deepseek_judge_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

13. SLM as an Agent (LangChain + Pydantic)

In [ ]:
!pip install -q langchain langgraph pydantic

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class MathAgentState(BaseModel):
    question: str
    plan: Optional[str] = None
    code: Optional[str] = None
    execution_result: Optional[str] = None
    final_answer: Optional[str] = None
    error: Optional[str] = None

In [ ]:
def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 300) -> str:
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=False,
        temperature=0.7,
        top_p=0.9,
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

In [ ]:
import re

def extract_section(text: str, header: str) -> str:
    pattern = rf"{header}:\s*(.*?)(?=\n[A-Z][A-Za-z ]*:\s|$)"
    match = re.search(pattern, text, re.DOTALL)
    return match.group(1).strip() if match else ""

def extract_code_block(text: str) -> str:
    # Regex to find a markdown code block for python
    # It looks for ```python then captures everything until ```
    patterns = [
        r"```python\s*(.*?)```",
        r"```py\s*(.*?)```",
        r"```\s*(.*?)```",
    ]
    for p in patterns:
      match = re.search(p, text, re.DOTALL | re.IGNORECASE)
      if match:
        return match.group(1).strip()

    return ""

def plan_and_code_node(state: MathAgentState) -> MathAgentState:
    prompt = f"""
You are a helpful math tutor for high school students.

Solve the following problem by producing:
Plan:
- short step-by-step reasoning

Code:
```python
- valid Python code that computes the answer
- print only the final numeric result

Question:
{state.question}
"""

    raw = generate_response(model, tokenizer, prompt)

    plan = extract_section(raw, "Plan")
    code = extract_code_block(raw) # Use the new function to extract code block

    if not code:
        return state.model_copy(update={
            "plan": plan or raw,
            "error": "No code generated by model."
        })

    if not plan:
      plan = "No explicit plan generated."

    return state.model_copy(update={
        "plan": plan,
        "code": code,
    })

In [ ]:
import io
import contextlib

def python_execute(code: str) -> str:
    buffer = io.StringIO()
    local_vars = {}

    try:
        with contextlib.redirect_stdout(buffer):
            exec(code, {}, local_vars)
        output = buffer.getvalue().strip()
        return output if output else "Code ran successfully but produced no printed output."
    except Exception as e:
        return f"ExecutionError: {str(e)}"

In [ ]:
def execute_code_node(state: MathAgentState) -> MathAgentState:
    if not state.code:
        return state.model_copy(update={"error": "No code available to execute."})

    result = python_execute(state.code)

    return state.model_copy(update={
        "execution_result": result
    })

In [ ]:
def final_answer_node(state: MathAgentState) -> MathAgentState:
    if state.error:
        prompt = f"""
You are a math tutor for high school students.

The earlier attempt had this issue:
{state.error}

Question:
{state.question}

Give a short, student-friendly answer anyway.
"""
    else:
        prompt = f"""
You are a math tutor for high school students.

Use the reasoning and execution result below to answer clearly.

Question:
{state.question}

Plan:
{state.plan}

Execution result:
{state.execution_result}

Write a concise, student-friendly final answer.
"""

    final = generate_response(model, tokenizer, prompt)

    return state.model_copy(update={
        "final_answer": final
    })

In [ ]:
from langgraph.graph import StateGraph, END

graph_builder = StateGraph(MathAgentState)

graph_builder.add_node("plan_and_code", plan_and_code_node)
graph_builder.add_node("execute_code", execute_code_node)
graph_builder.add_node("final_answer", final_answer_node)

graph_builder.set_entry_point("plan_and_code")
graph_builder.add_edge("plan_and_code", "execute_code")
graph_builder.add_edge("execute_code", "final_answer")
graph_builder.add_edge("final_answer", END)

math_agent = graph_builder.compile()

In [ ]:
initial_state = MathAgentState(
    question="Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
)

result = math_agent.invoke(initial_state)

print("PLAN:\n", result['plan'])
print("\nCODE:\n", result['code'])
print("\nEXECUTION RESULT:\n", result['execution_result'])
print("\nFINAL ANSWER:\n", result['final_answer'])

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PLAN:
 ```python
# Define variables
april_sales = 48 # Number of clips sold in April
may_sales = None # Number of clips sold in May
total_clips = None # Total number of clips sold in April and May

# Calculate may sales
may_sales = april_sales / 2

# Calculate total sales
total_clips = april_sales + may_sales

print(total_clips) # Output: The final answer
```

CODE:
 # Define variables
april_sales = 48 # Number of clips sold in April
may_sales = None # Number of clips sold in May
total_clips = None # Total number of clips sold in April and May

# Calculate may sales
may_sales = april_sales / 2

# Calculate total sales
total_clips = april_sales + may_sales

print(total_clips) # Output: The final answer

EXECUTION RESULT:
 72.0

FINAL ANSWER:
 First find how many clips Natalia sold in May: 48 clips * .5 = <<48*.5=24>>24 clips
Then add that number to the number sold in April to find the total: 24 clips + 48 clips = <<24+48=72>>72 clips
#### 72
